In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score

In [2]:
# carregando arquivo csv
base = pd.read_csv('/content/enem_sample10k.csv')

<ipython-input-2-3e290aba5f99>:2: DtypeWarning: Columns (136,137) have mixed types. Specify dtype option on import or set low_memory=False.
  base = pd.read_csv('/content/enem_sample10k.csv')


In [3]:
base.describe()

,HASHID,NU_INSCRICAO,NU_ANO,CO_MUNICIPIO_RESIDENCIA,CO_UF_RESIDENCIA,NU_IDADE,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,CO_MUNICIPIO_NASCIMENTO,...,TP_STATUS_REDACAO,NU_NOTA_COMP1,NU_NOTA_COMP2,NU_NOTA_COMP3,NU_NOTA_COMP4,NU_NOTA_COMP5,NU_NOTA_REDACAO,Q005,IN_TEMPO_ADICIONAL,TP_FAIXA_ETARIA
count,9.913000e+03,9.913000e+03,9913.000000,6.401000e+03,6401.000000,6401.000000,9792.000000,9913.000000,9913.000000,6.185000e+03,...,6504.000000,6504.000000,6504.000000,6504.000000,6504.000000,6504.000000,6504.000000,9866.000000,3087.000000,3512.000000
mean,-4.056898e+15,1.902057e+11,2019.019974,3.104965e+06,30.896735,22.150758,0.768689,2.143852,1.033996,3.077898e+06,...,1.126230,120.811808,114.126691,106.374539,119.852399,84.710947,545.876384,3.837219,0.002268,6.251139
std,5.300056e+18,8.293488e+09,0.829615,1.000319e+06,9.973800,7.433450,0.620282,1.018932,0.218574,9.817975e+05,...,0.749941,33.257031,49.237207,45.038831,37.511100,57.555483,195.779788,1.485956,0.047573,4.150353
min,-9.223324e+18,1.800072e+11,2018.000000,1.100023e+06,11.000000,14.000000,0.000000,0.000000,0.000000,1.100023e+06,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
25%,-4.588593e+18,1.800120e+11,2018.000000,2.412401e+06,24.000000,18.000000,0.000000,1.000000,1.000000,2.408102e+06,...,1.000000,100.000000,100.000000,80.000000,100.000000,40.000000,420.000000,3.000000,0.000000,3.000000
50%,-5.215316e+16,1.900038e+11,2019.000000,3.131307e+06,31.000000,19.000000,1.000000,2.000000,1.000000,3.116902e+06,...,1.000000,120.000000,120.000000,120.000000,120.000000,80.000000,560.000000,4.000000,0.000000,5.000000
75%,4.625001e+18,2.000028e+11,2020.000000,3.550308e+06,35.000000,24.000000,1.000000,3.000000,1.000000,3.550308e+06,...,1.000000,140.000000,140.000000,120.000000,140.000000,120.000000,660.000000,5.000000,0.000000,10.000000
max,9.223215e+18,2.000068e+11,2020.000000,5.300108e+06,53.000000,71.000000,4.000000,5.000000,4.000000,5.300108e+06,...,9.000000,200.000000,200.000000,200.000000,200.000000,200.000000,980.000000,20.000000,1.000000,19.000000


In [4]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9913 entries, 0 to 9912
Columns: 140 entries, HASHID to TP_FAIXA_ETARIA
dtypes: float64(81), int64(16), object(43)
memory usage: 10.6+ MB


In [5]:
base.columns

Index(['HASHID', 'NU_INSCRICAO', 'NU_ANO', 'CO_MUNICIPIO_RESIDENCIA',
       'NO_MUNICIPIO_RESIDENCIA', 'CO_UF_RESIDENCIA', 'SG_UF_RESIDENCIA',
       'NU_IDADE', 'TP_SEXO', 'TP_ESTADO_CIVIL',
       ...
       'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'Q026', 'Q027',
       'IN_TEMPO_ADICIONAL', 'TP_FAIXA_ETARIA'],
      dtype='object', length=140)

## 1. Qual a distribuição dos participantes por estado onde a prova foi realizada em cada ano? E no total do Brasil?



## P: Questão teste: Qual ano e estado que tiveram mais provas aplicadas?
## R: SP em 2020, com 567

In [6]:
# Distribuição por estado e ano
distrib_est_ano = base.groupby(['NU_ANO', 'SG_UF_PROVA']).size().reset_index(name='Qtd_Provas')

In [7]:
distrib_est_ano

,NU_ANO,SG_UF_PROVA,Qtd_Provas
0,2018,AC,24
1,2018,AL,66
2,2018,AM,71
3,2018,AP,37
4,2018,BA,263
...,...,...,...
76,2020,RS,134
77,2020,SC,86
78,2020,SE,53
79,2020,SP,567


In [8]:
# Estado e ano com mais provas aplicadas
mais_provas = distrib_est_ano.sort_values('Qtd_Provas', ascending=False).head(1)

In [9]:
mais_provas

,NU_ANO,SG_UF_PROVA,Qtd_Provas
79,2020,SP,567


In [10]:
# Distribuição total por estado (todas as edições)
distrib_total_estado = base.groupby('SG_UF_PROVA').size().sort_values(ascending=False)

In [11]:
distrib_total_estado

,0
SG_UF_PROVA,
SP,1610
MG,977
BA,754
RJ,677
CE,569
PE,551
PA,542
RS,415
MA,401


In [15]:
# Distribuição por estado e ano
distrib_ano = base.groupby(['NU_ANO']).size().reset_index(name='Qtd_Provas_Ano')

In [16]:
distrib_ano

,NU_ANO,Qtd_Provas_Ano
0,2018,3314
1,2019,3087
2,2020,3512


## Considerando apenas 2018, quais as métricas globais de média, mediana, primeiro quartil (25%), terceiro quartil (75%) dos participantes em matemática (NU_NOTA_MT)?
Obs.: Considerar apenas quem participou da prova, TP_PRESENCA_MT = 1


In [12]:
# Filtrar 2018 e quem fez a prova de matemática
base_2018 = base[(base['NU_ANO'] == 2018) & (base['TP_PRESENCA_MT'] == 1)]
base_2018

,HASHID,NU_INSCRICAO,NU_ANO,CO_MUNICIPIO_RESIDENCIA,NO_MUNICIPIO_RESIDENCIA,CO_UF_RESIDENCIA,SG_UF_RESIDENCIA,NU_IDADE,TP_SEXO,TP_ESTADO_CIVIL,...,Q020,Q021,Q022,Q023,Q024,Q025,Q026,Q027,IN_TEMPO_ADICIONAL,TP_FAIXA_ETARIA
1918,6286937377212396751,180013342985,2018,3525904.0,Jundiaí,35.0,SP,17.0,F,0.0,...,B,B,C,A,B,B,C,D,NaN,NaN
2325,-492638147235987984,180010800610,2018,2302800.0,Canindé,23.0,CE,17.0,F,0.0,...,A,A,B,A,A,A,B,A,NaN,NaN
2378,4722630572071719439,180011634605,2018,3505500.0,Barretos,35.0,SP,32.0,F,0.0,...,A,A,B,A,B,B,A,A,NaN,NaN
2769,-2294552713208492330,180013029170,2018,5300108.0,Brasília,53.0,DF,17.0,F,0.0,...,A,B,C,A,B,B,B,D,NaN,NaN
2880,-6572758433719613443,180007993784,2018,3106200.0,Belo Horizonte,31.0,MG,15.0,F,0.0,...,B,B,C,B,C,B,C,D,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7543,4194506909497725150,180009141573,2018,3147006.0,Paracatu,31.0,MG,25.0,M,0.0,...,A,A,E,B,B,B,A,A,NaN,NaN
7545,-9155803599523195945,180011232200,2018,3509502.0,Campinas,35.0,SP,17.0,F,0.0,...,B,B,E,B,D,B,B,D,NaN,NaN
7546,-4011331330885547240,180011042599,2018,3509502.0,Campinas,35.0,SP,17.0,M,NaN,...,B,B,D,A,B,B,B,A,NaN,NaN
7547,6006111907907899425,180009090491,2018,5215231.0,Novo Gama,52.0,GO,37.0,F,1.0,...,A,A,B,B,B,B,A,A,NaN,NaN


In [34]:
# Métricas globais
metricas_nacionais = base_2018['NU_NOTA_MT'].describe()
terceiro_quartil_nacional = metricas_nacionais['75%']

In [35]:
terceiro_quartil_nacional

np.float64(600.05)

P: Considerando apenas 2018, quais as métricas globais de média, mediana, primeiro quartil (25%), terceiro quartil (75%) dos participantes em matemática (NU_NOTA_MT)?

In [19]:
metricas_nacionais

,NU_NOTA_MT
count,2278.000000
mean,534.340825
std,103.858679
min,0.000000
25%,454.025000
50%,515.250000
75%,600.050000
max,935.600000


P: : Qual o terceiro quartil Nacional em matemática?

In [20]:
terceiro_quartil_nacional

np.float64(600.05)

P: Faça a mesma análise de 2018, porém considerando quebras por estado (local da prova).
Indique quais estados tem métricas superiores e inferiores do que as métricas nacionais.


In [24]:
# Métricas por estado
metricas_estados = base_2018.groupby('SG_UF_PROVA')['NU_NOTA_MT'].describe()
metricas_estados



,count,mean,std,min,25%,50%,75%,max
SG_UF_PROVA,,,,,,,,
AC,22.0,492.645455,92.870656,375.4,411.200,485.50,553.775,726.2
AL,49.0,533.363265,98.817755,378.8,449.500,529.80,608.200,711.7
AM,42.0,497.771429,90.670823,377.8,427.400,488.10,532.650,745.5
AP,25.0,501.864000,80.321318,394.6,433.300,493.30,551.600,651.7
BA,184.0,499.982609,83.981415,370.6,436.275,486.65,542.925,801.2
CE,119.0,523.486555,96.355714,382.2,451.400,505.20,577.500,848.1
DF,41.0,541.560976,88.465586,386.5,484.400,528.60,570.400,734.4
ES,44.0,544.481818,104.412672,378.2,471.000,509.05,619.050,781.3
GO,90.0,543.768889,119.926944,372.9,452.950,530.30,618.950,869.8


In [26]:
estados_media_superior = metricas_estados[metricas_estados['mean'] > metricas_nacionais['mean']]
estados_media_superior

,count,mean,std,min,25%,50%,75%,max
SG_UF_PROVA,,,,,,,,
DF,41.0,541.560976,88.465586,386.5,484.400,528.60,570.400,734.4
ES,44.0,544.481818,104.412672,378.2,471.000,509.05,619.050,781.3
GO,90.0,543.768889,119.926944,372.9,452.950,530.30,618.950,869.8
MG,211.0,554.261611,105.565650,372.5,476.550,528.70,638.200,799.4
MS,23.0,557.547826,122.760724,384.7,491.350,549.80,612.550,935.6
PI,54.0,540.135185,105.832542,393.1,463.175,523.60,612.825,804.7
PR,87.0,563.950575,103.066355,388.5,480.050,544.60,650.050,810.8
RJ,156.0,554.105128,116.252862,0.0,464.300,546.60,633.625,823.6
RS,108.0,544.287963,99.651505,373.5,475.225,534.55,618.550,770.1


In [27]:
estados_media_inferiores = metricas_estados[metricas_estados['mean'] < metricas_nacionais['mean']]
estados_media_inferiores

,count,mean,std,min,25%,50%,75%,max
SG_UF_PROVA,,,,,,,,
AC,22.0,492.645455,92.870656,375.4,411.200,485.50,553.775,726.2
AL,49.0,533.363265,98.817755,378.8,449.500,529.80,608.200,711.7
AM,42.0,497.771429,90.670823,377.8,427.400,488.10,532.650,745.5
AP,25.0,501.864000,80.321318,394.6,433.300,493.30,551.600,651.7
BA,184.0,499.982609,83.981415,370.6,436.275,486.65,542.925,801.2
CE,119.0,523.486555,96.355714,382.2,451.400,505.20,577.500,848.1
MA,79.0,493.475949,101.681664,0.0,433.200,483.20,544.800,822.3
MT,36.0,516.308333,96.868333,405.0,441.875,489.70,573.900,755.9
PA,120.0,503.036667,96.753871,375.0,439.650,482.75,534.450,779.1


Qual o terceiro quartil de Roraima? Como está RR comparado com BR?


In [28]:
# Comparar terceiro quartil de Roraima (RR) com nacional
q3_rr = metricas_estados.loc['RR', '75%']
comparacao_rr = 'acima' if q3_rr > terceiro_quartil_nacional else 'abaixo'

In [29]:
q3_rr

np.float64(448.65)

In [30]:
comparacao_rr

'abaixo'

4. Faça a mesma análise de 2018, porém considerando escolaridade do pai (Q001). Indique quais casos possuem
métricas superiores e inferiores do que as métricas nacionais.
Questão teste: Qual a mediana dos participantes cujos pais completaram o ensino médio e qual a mediada
quando os pais concluíram a faculdade?


In [37]:
# Agrupar por escolaridade do pai
metricas_pai = base_2018.groupby('Q001')['NU_NOTA_MT'].describe()

# Mediana de ensino médio completo ('D') e faculdade completa ('H')
mediana_e = metricas_pai.loc['E', '50%']  # Ensino Médio Completo
mediana_f = metricas_pai.loc['F', '50%']  # Ensino Superior Completo


In [38]:
mediana_e

np.float64(537.1)

In [39]:
mediana_f

np.float64(620.4)

In [40]:
metricas_pai

,count,mean,std,min,25%,50%,75%,max
Q001,,,,,,,,
A,105.0,478.440952,90.258257,0.0,420.000,461.90,534.200,707.9
B,486.0,499.743004,80.686582,369.4,435.725,488.90,546.825,782.4
C,314.0,515.207643,100.735407,0.0,441.500,494.65,573.700,814.2
D,251.0,525.952590,88.797061,372.5,459.500,513.70,581.350,803.3
E,622.0,552.115113,100.144119,377.4,476.925,537.10,626.925,833.3
F,193.0,618.598964,115.914417,373.1,529.600,620.40,711.700,935.6
G,111.0,619.346847,123.905139,378.5,513.850,631.50,721.550,869.8
H,196.0,503.954082,82.285065,370.6,443.825,496.65,547.850,759.8
